In [1]:
from trials_pipeline import fetch_trials, extract_fields, run_pipeline, build_include_exclude_patterns, classify
import pandas as pd

In [2]:
window1 = run_pipeline("kidney disease", start_date="2020-01-01", end_date="2023-12-31")
window2 = run_pipeline("kidney disease", start_date="2023-07-01", end_date="2026-07-01")

Fetching (MeSH/native query.cond): kidney disease
  828 unique studies found
  Saved to kidney_disease_2020-01-01_to_2023-12-31_trials.csv
Fetching (MeSH/native query.cond): kidney disease
  668 unique studies found
  Saved to kidney_disease_2023-07-01_to_2026-07-01_trials.csv


In [3]:
import pandas as pd

df1 = pd.read_csv("kidney_disease_2020-01-01_to_2023-12-31_trials.csv", keep_default_na=False)
df1["date_window"] = "2020-01-01 to 2023-12-31"

df2 = pd.read_csv("kidney_disease_2023-07-01_to_2026-07-01_trials.csv", keep_default_na=False)
df2["date_window"] = "2023-07-01 to 2026-07-01"

df_all = pd.concat([df1, df2], ignore_index=True)
df_all = df_all[df_all["phase"] != "NA"]

In [4]:
lookup = pd.read_csv("industry_sponsor_normalization_USABLE_LOOKUP.csv")
df_all["lead_sponsor_normalized"] = df_all["lead_sponsor"].str.strip().str.lower()
lookup["raw_sponsor_name_normalized"] = lookup["raw_sponsor_name"].str.strip().str.lower()
df_all = df_all.merge(
    lookup[["raw_sponsor_name_normalized", "dashboard_sponsor_name"]],
    left_on="lead_sponsor_normalized",
    right_on="raw_sponsor_name_normalized",
    how="left",
    indicator=True
)
df_all["dashboard_sponsor_name"] = df_all["dashboard_sponsor_name"].fillna(df_all["lead_sponsor"])

In [5]:
from trials_pipeline import build_include_exclude_patterns, classify

mesh_dict = pd.read_csv("mesh_include_exclude_dictionary.csv")
mesh_dict = mesh_dict[mesh_dict["Disease Area"] == "kidney"]

# The Include rows use slash-separated synonyms as human-readable labels
# (e.g. "Chronic kidney disease / CKD / chronic renal failure") — these need
# expansion into individual keywords, same issue we hit with the CV dictionary.
kidney_include_keywords = {
    "CKD": ["chronic kidney disease", "ckd", "chronic renal failure", "chronic renal insufficiency", "renal insufficiency"],
    "ESRD": ["esrd", "end stage renal disease", "end-stage renal disease", "end stage kidney disease", "eskd"],
    "AKI": ["acute kidney injury", "aki", "acute renal failure"],
    "Glomerular disease": ["glomerulonephritis", "glomerulosclerosis", "glomerulopathy", "iga nephropathy",
                            "focal segmental glomerulosclerosis", "fsgs", "membranous nephropathy",
                            "minimal change disease", "lupus nephritis", "nephrotic syndrome", "nephritis"],
    "Diabetic kidney disease": ["diabetic nephropathy", "diabetic kidney disease", "dkd"],
    "Polycystic kidney disease": ["polycystic kidney", "adpkd", "arpkd"],
    "Transplant": ["kidney transplant", "renal transplant", "transplant rejection"],
    "aHUS/complement": ["atypical hemolytic uremic syndrome", "ahus", "complement 3 glomerulopathy",
                         "c3g", "dense deposit disease", "membranoproliferative glomerulonephritis",
                         "mpgn", "thrombotic microangiopathy"],
    "ANCA/anti-GBM": ["anca-associated vasculitis", "anca-associated glomerulonephritis",
                       "anti-gbm", "goodpasture"],
    "CKD complications": ["anemia of chronic kidney disease", "renal anemia", "secondary hyperparathyroidism",
                           "hyperphosphatemia", "hyperkalemia", "ckd-associated pruritus", "uremic pruritus"],
    "Hereditary": ["alport syndrome", "nephropathic cystinosis", "apol1-mediated kidney disease",
                   "primary hyperoxaluria"],
    "Hypertensive kidney disease": ["hypertensive nephropathy", "hypertensive kidney disease"],
    "Renal impairment": ["renal impairment", "renal dysfunction", "kidney impairment"],
}
kidney_include_keywords["Additional terms"] = [
    "kidney disease", "kidney failure", "renal disease", "renal failure", "kidney injury",
    "immunoglobulin a nephropathy", "igan", "hemodialysis", "dialysis",
    "proteinuria", "proteinuric kidney", "glomerular disease", "hypophosphatemia",
    "bk virus nephropathy", "renal artery", "glomerular basement membrane",
    "nephropath", "complement-mediated kidney", "complement-mediated renal",
    "anti-neutrophil cytoplasmic antibody", "polyangiitis", "calciphylaxis", "hyperkalaemia", "impaired renal function",
]

include_pattern, exclude_pattern = build_include_exclude_patterns(mesh_dict, kidney_include_keywords)

df_all["indication_classification"] = df_all["conditions"].apply(
    lambda c: classify(c, include_pattern, exclude_pattern, match_label="true_kidney")
)
df_all["is_drug_biologic"] = df_all["intervention_types"].str.contains("DRUG|BIOLOGICAL", na=False)

In [6]:
df_all["_merge"].value_counts()

_merge
both          781
left_only     528
right_only      0
Name: count, dtype: int64

In [7]:
df_all[df_all["_merge"] == "left_only"]["lead_sponsor"].value_counts().to_csv("kidney_unmapped_sponsors.csv")

In [8]:
review_list = (
    df_all[df_all["indication_classification"] == "needs_review"]["conditions"]
    .str.split("|").explode().value_counts()
)
review_list.to_csv("kidney_conditions_needing_review.csv")

In [9]:
df_phased = df_all[~df_all["phase"].isin(["BLANK", "PHASE2,PHASE3"])]

In [10]:
df_observational = df_all[
    (df_all["study_type"] == "OBSERVATIONAL")
    & (df_all["indication_classification"] == "true_kidney")
]
df_true_kidney_drug = df_phased[
    (df_phased["indication_classification"] == "true_kidney")
    & (df_phased["is_drug_biologic"])
]

## Comany and indication views

In [11]:
company_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2"]
company_late_phases = ["PHASE3", "PHASE4"]
indication_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2", "PHASE2"]
indication_late_phases = ["PHASE3", "PHASE4"]

df_company_early = df_true_kidney_drug[df_true_kidney_drug["phase"].isin(company_early_phases)]
df_company_late = df_true_kidney_drug[df_true_kidney_drug["phase"].isin(company_late_phases)]

top20_companies_all_phases = (
    df_true_kidney_drug.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_companies_early_phase = (
    df_company_early.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_by_window = (
    df_true_kidney_drug.groupby(["date_window", "dashboard_sponsor_name"])["nct_id"]
    .nunique().reset_index()
    .sort_values(["date_window", "nct_id"], ascending=[True, False])
    .groupby("date_window").head(20)
)

df_indications = df_true_kidney_drug.assign(
    conditions=df_true_kidney_drug["conditions"].str.split("|")
).explode("conditions")

df_indications.to_csv("kidney_conditions_exploded_for_tableau.csv", index=False)

top_indications_early = (
    df_indications[df_indications["phase"].isin(indication_early_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)
top_indications_late = (
    df_indications[df_indications["phase"].isin(indication_late_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)

## Kidney File Exports

In [12]:
top20_companies_all_phases.to_csv("kidney_top20_companies_all_phases.csv")
top20_companies_early_phase.to_csv("kidney_top20_companies_early_phase.csv")
top20_by_window.to_csv("kidney_top20_companies_by_date_window.csv", index=False)
top_indications_early.to_csv("kidney_top_indications_early_phase.csv")
top_indications_late.to_csv("kidney_top_indications_late_phase.csv")
df_observational.to_csv("kidney_observational_studies.csv", index=False)

## Tableau Export

In [13]:
df_all.to_csv("kidney_full_dataset_for_tableau.csv", index=False)

## Explode Collaborators

In [14]:
df_collab = df_all.assign(
    collaborators=df_all["collaborators"].str.split("|")
).explode("collaborators")
df_collab = df_collab[df_collab["collaborators"].notna() & (df_collab["collaborators"] != "")]
df_collab.to_csv("kidney_collaborations.csv", index=False)

In [16]:
df_indications["conditions"].isna().sum()


np.int64(0)

In [17]:
df_indications[df_indications["conditions"].isna()].head(10)

,nct_id,title,seed_term,study_type,phase,status,first_posted,last_update_posted,lead_sponsor,lead_sponsor_class,...,collaborators,drug_names,intervention_types,date_window,lead_sponsor_normalized,raw_sponsor_name_normalized,dashboard_sponsor_name,_merge,indication_classification,is_drug_biologic


In [18]:
len(df_indications)

1059